# NMF on ModulAir 00605

In [1]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [2]:
#importing data from Modulair MOD-00605
data = pd.read_csv('MOD-00605.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-31T05:21:09Z,577082339,2025-12-31T00:21:09Z,MOD-00605,44.1,-2.7,1.034,0.117,0.016,0.006,0.011,...,NaN,NaN,16584,16585,16586,NaN,NaN,NaN,NaN,10.87
2025-12-31T05:20:20Z,577082337,2025-12-31T00:20:20Z,MOD-00605,44.7,-2.8,1.267,0.144,0.043,0.000,0.013,...,NaN,NaN,16584,16585,16586,NaN,NaN,NaN,NaN,8.18
2025-12-31T05:19:20Z,577082335,2025-12-31T00:19:20Z,MOD-00605,44.6,-2.8,0.841,0.138,0.013,0.013,0.000,...,NaN,NaN,16584,16585,16586,NaN,NaN,NaN,NaN,10.31
2025-12-31T05:18:20Z,577082336,2025-12-31T00:18:20Z,MOD-00605,44.4,-2.9,0.610,0.108,0.007,0.000,0.000,...,NaN,NaN,16584,16585,16586,NaN,NaN,NaN,NaN,8.64
2025-12-30T03:39:14Z,576340801,2025-12-29T22:39:14Z,MOD-00605,44.3,0.5,1.078,0.210,0.030,0.003,0.015,...,NaN,NaN,16584,16585,16586,NaN,NaN,NaN,NaN,15.49


In [5]:
start = data.index.min()
end = data.index.max()

print(start, end)

2025-04-15T12:29:01Z 2025-12-31T05:21:09Z


In [6]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-31T05:21:09Z,2025-12-31T00:21:09Z,NaN,NaN,NaN,NaN,1.034,0.117,0.016,0.006,0.011,0.000
2025-12-31T05:20:20Z,2025-12-31T00:20:20Z,NaN,NaN,NaN,NaN,1.267,0.144,0.043,0.000,0.013,0.007
2025-12-31T05:19:20Z,2025-12-31T00:19:20Z,NaN,NaN,NaN,NaN,0.841,0.138,0.013,0.013,0.000,0.000
2025-12-31T05:18:20Z,2025-12-31T00:18:20Z,NaN,NaN,NaN,NaN,0.610,0.108,0.007,0.000,0.000,0.007
2025-12-30T03:39:14Z,2025-12-29T22:39:14Z,NaN,NaN,NaN,NaN,1.078,0.210,0.030,0.003,0.015,0.010


In [7]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31T00:21:09Z,NaN,NaN,NaN,NaN,1.034,0.117,0.016,0.006,0.011,0.000
1,2025-12-31T00:20:20Z,NaN,NaN,NaN,NaN,1.267,0.144,0.043,0.000,0.013,0.007
2,2025-12-31T00:19:20Z,NaN,NaN,NaN,NaN,0.841,0.138,0.013,0.013,0.000,0.000
3,2025-12-31T00:18:20Z,NaN,NaN,NaN,NaN,0.610,0.108,0.007,0.000,0.000,0.007
4,2025-12-29T22:39:14Z,NaN,NaN,NaN,NaN,1.078,0.210,0.030,0.003,0.015,0.010


In [8]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31 00:21:09,NaN,NaN,NaN,NaN,1.034,0.117,0.016,0.006,0.011,0.000
1,2025-12-31 00:20:20,NaN,NaN,NaN,NaN,1.267,0.144,0.043,0.000,0.013,0.007
2,2025-12-31 00:19:20,NaN,NaN,NaN,NaN,0.841,0.138,0.013,0.013,0.000,0.000
3,2025-12-31 00:18:20,NaN,NaN,NaN,NaN,0.610,0.108,0.007,0.000,0.000,0.007
4,2025-12-29 22:39:14,NaN,NaN,NaN,NaN,1.078,0.210,0.030,0.003,0.015,0.010


In [9]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
2,2025-04-15 10:00:00,1602.06,41.17,92.37,2.26,20.74,2.23,0.59,0.14,0.14,0.09
3,2025-04-15 11:00:00,1550.78,42.50,86.12,4.06,21.48,2.43,0.65,0.16,0.15,0.10
4,2025-04-15 12:00:00,1353.58,16.95,88.56,5.95,9.35,1.03,0.34,0.09,0.10,0.06
5,2025-04-15 13:00:00,1004.71,7.61,90.57,4.18,5.46,0.57,0.21,0.06,0.07,0.05
6,2025-04-15 14:00:00,888.54,7.88,86.08,3.24,2.81,0.34,0.14,0.04,0.06,0.05
...,...,...,...,...,...,...,...,...,...,...,...
6038,2025-12-27 16:00:00,716.04,32.70,58.77,1.90,2.59,0.44,0.13,0.03,0.03,0.02
6039,2025-12-27 17:00:00,737.84,33.36,58.69,1.79,2.78,0.50,0.15,0.03,0.03,0.01
6040,2025-12-27 18:00:00,735.48,32.40,59.04,1.82,2.59,0.45,0.13,0.02,0.02,0.01
6041,2025-12-27 19:00:00,715.52,32.42,59.37,1.64,3.00,0.48,0.13,0.02,0.03,0.01


In [10]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-04-15 10:00:00,1602.06,41.17,92.37,2.26,20.74,2.23,0.59,0.14,0.14,0.09
2025-04-15 11:00:00,1550.78,42.50,86.12,4.06,21.48,2.43,0.65,0.16,0.15,0.10
2025-04-15 12:00:00,1353.58,16.95,88.56,5.95,9.35,1.03,0.34,0.09,0.10,0.06
2025-04-15 13:00:00,1004.71,7.61,90.57,4.18,5.46,0.57,0.21,0.06,0.07,0.05
2025-04-15 14:00:00,888.54,7.88,86.08,3.24,2.81,0.34,0.14,0.04,0.06,0.05


In [11]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [12]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [13]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-04-15 10:00:00,0.333031,0.658299,0.760810,0.034367,0.268757,0.089992,0.040383,0.032037,0.040462,0.109756
2025-04-15 11:00:00,0.322371,0.679565,0.709332,0.061740,0.278347,0.098063,0.044490,0.036613,0.043353,0.121951
2025-04-15 12:00:00,0.281377,0.271027,0.729429,0.090481,0.121161,0.041566,0.023272,0.020595,0.028902,0.073171
2025-04-15 13:00:00,0.208856,0.121682,0.745985,0.063564,0.070753,0.023002,0.014374,0.013730,0.020231,0.060976
2025-04-15 14:00:00,0.184707,0.125999,0.709003,0.049270,0.036413,0.013721,0.009582,0.009153,0.017341,0.060976


In [14]:
data.to_csv(r'MOD-000605_timeseries_hourly_scaled.csv')

## Implementing NMF

In [15]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [16]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-04-15 10:00:00,0.275734,0.665009,0.775422,0.052070,0.248644,0.124534,0.057167,0.036688,0.038416,0.096860
2025-04-15 11:00:00,0.270530,0.685817,0.724875,0.050650,0.254459,0.136729,0.064331,0.042325,0.044295,0.105726
2025-04-15 12:00:00,0.196652,0.287981,0.745399,0.039055,0.134396,0.034762,0.018366,0.013386,0.019151,0.081957
2025-04-15 13:00:00,0.168041,0.132232,0.749625,0.034392,0.100392,0.003407,0.003984,0.004163,0.011084,0.074626
2025-04-15 14:00:00,0.159949,0.135509,0.705361,0.032700,0.093571,0.003143,0.003675,0.003840,0.010278,0.069448
...,...,...,...,...,...,...,...,...,...,...
2025-12-27 16:00:00,0.179361,0.518200,0.476796,0.035132,0.038262,0.008062,0.004985,0.004051,0.006289,0.029314
2025-12-27 17:00:00,0.181440,0.529362,0.476032,0.035485,0.042666,0.009182,0.003758,0.002109,0.003263,0.019830
2025-12-27 18:00:00,0.179280,0.514300,0.479297,0.035154,0.040484,0.006439,0.002384,0.001151,0.002368,0.018879


In [17]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.041045,0.059209,0.100669,0.025294
1,0.034104,0.063958,0.104856,0.031256
2,0.062311,0.014915,0.037344,0.012420
3,0.072708,0.000000,0.010835,0.006078
4,0.067768,0.000000,0.012116,0.005606
...,...,...,...,...
5811,0.020729,0.002973,0.081961,0.004495
5812,0.019932,0.004671,0.083908,0.000849
5813,0.021234,0.003444,0.081295,0.000036
5814,0.021402,0.004671,0.081120,0.000111


In [18]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [19]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,2.061312,0.904765,9.806939,0.425397,1.380754,0.000000,0.000000,0.000000,0.076392,0.820688
Feature 2,0.402470,0.226329,0.621694,0.041184,3.242272,1.863807,0.685438,0.326993,0.207157,0.000000
Feature 3,1.649835,6.080397,3.307866,0.319568,0.000000,0.000000,0.000000,0.000000,0.000000,0.016793
Feature 4,0.047827,0.093394,0.121900,0.000000,0.000000,0.560627,0.655604,0.685037,0.909918,2.430829


In [20]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-04-15 10:00:00,0.041045,0.059209,0.100669,0.025294
2025-04-15 11:00:00,0.034104,0.063958,0.104856,0.031256
2025-04-15 12:00:00,0.062311,0.014915,0.037344,0.012420
2025-04-15 13:00:00,0.072708,0.000000,0.010835,0.006078
2025-04-15 14:00:00,0.067768,0.000000,0.012116,0.005606
...,...,...,...,...
2025-12-27 16:00:00,0.020729,0.002973,0.081961,0.004495
2025-12-27 17:00:00,0.019932,0.004671,0.083908,0.000849
2025-12-27 18:00:00,0.021234,0.003444,0.081295,0.000036


In [21]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,2.061312,0.904765,9.806939,0.425397,1.380754,0.000000,0.000000,0.000000,0.076392,0.820688
Factor 2,0.402470,0.226329,0.621694,0.041184,3.242272,1.863807,0.685438,0.326993,0.207157,0.000000
Factor 3,1.649835,6.080397,3.307866,0.319568,0.000000,0.000000,0.000000,0.000000,0.000000,0.016793
Factor 4,0.047827,0.093394,0.121900,0.000000,0.000000,0.560627,0.655604,0.685037,0.909918,2.430829


In [22]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.361560,0.053397,0.572036,0.005092,0.007914
no,0.387186,0.028353,0.574955,0.000000,0.009505
no2,0.068853,0.013028,0.914670,0.004314,-0.000865
o3,0.580748,0.027847,0.387212,0.004382,-0.000188
bin0,0.364661,0.647696,0.000000,0.000000,-0.012357
bin1,0.000000,0.868644,0.000000,0.209681,-0.078326
bin2,0.000000,0.661428,0.000000,0.507692,-0.169120
bin3,0.000000,0.425260,0.000000,0.714948,-0.140208
bin4,0.097476,0.199937,0.000000,0.704760,-0.002173
bin5,0.354942,0.000000,0.014357,0.638154,-0.007453


In [23]:
results.to_csv(r'4_factor_results.csv')
comp.to_csv(r'4_factor_comp.csv')
res.to_csv(r'4_factor_resid.csv')

In [24]:
len(data)

5816